# Erstellung der gocfl create Befehle

Dieses Jupyiter Notebook erstellt die gocfl create-Befehle für eine bestimmte Collection. 
Die Doku für den Aufbau eines gocfl create Befehls befindet sich hier: https://github.com/je4/gocfl/blob/main/docs/create.md

## Vorbereitung

Die AIPs liegen (mehrheitlich) als ZIP-File im Ordner Objects, im Unterordner der jeweiligen Signatur. Es macht keinen Sinn, ZIP-Files in ein ZIP-Archiv zu ingesten. Daher müssen die ZIP-Files vorher entzippt und die Ordner danach bereinigt werden (ZIP-Files löschen). Dabei können auch Validierungen, Migrationen und Virenüberprüfung stattfinden, da dies zum Zeitpunkt alles manuelle Prozesse sind. 

Dieses Notebook geht davon aus, dass im Unterordner objects/{signature}/ ein weiterer Unterordner liegt, dessen Content ins Archiv gelagert wird. Ist dies nicht der Fall, wird das Script abgebrochen.  
    

## config.py

In der Config wird die aktuell zu verarbeitende Collection sowie diverse Dateipfade konfiguriert. Bspw. für E-Manuscripta:

    collection_id = 'zhb_e-manuscripta'
    dlza_root = 'd:/Ingest'
    gocfl_conf = 'd:/Ingest/config/zhb-config.toml'

## signature

Die Signature ist zentral für die Erstellung des storage roots, sowie das Auffinden der Objekt-Pfade, Metadaten und Info-Dateien. Grundsätzlich sollte für jede Collection eine Textdatei namens '/signatures.txt' mit den signatures vorliegen. Diese werden entweder durch ein anderes Jupyter Notebook konfiguriert oder können von Hand erstellt werden.
Der Dateipfad kann konfiguriert werden. 

## storage root

Hier wird davon ausgegangen, dass der storage root ein ZIP file sein soll. Für jede signature wird ein storage_root angelegt. Der Root-Path kann konfiguriert werden. 

## update

Fürs Testen muss nach dem Durchlauf des Scripts der storage root Ordner wieder geleert werden, ansonsten geht das script davon aus, dass das Archiv (storage root) bereits besteht und ein Update-Workflow notwendig ist. Der Update-Workflow wird in einem anderen Notebook beschrieben. Für den Fall, dass das Archiv bereits besteht, wird die signature in ein update-file geschrieben. 


###  Create Befehl für gocfl generieren

Die gocfl create Befehle für alle Signaturen werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt. Der Pfad für die Config.toml kann konfiguriert werden.  

Muster:

    gocfl create ./archiv.zip ./object-directory metadata:./metadata-directory --config ./config/gocfl.toml -i 'signature'  --ext-NNNN-metafile-source ./info.json


In [1]:
import config
import os
from zipfile import ZipFile
import shutil
from datetime import datetime
from pathlib import Path


#prepare archive structure


root = config.dlza_root
coll = config.collection_id
org = config.organisation_id
files = config.files_path
metadata =  f'{coll}/{config.metadata_path}'
info = f'{coll}/{config.info_path}'
objects = f'{coll}/{config.object_path}'
gocfl_conf = config.gocfl_conf

# filenames
f_signatures = f'{coll}/{files}/{coll}_signatures.txt'
f_error = f'{coll}/{files}/{coll}_create_errors.txt'
f_update = f'{coll}/{files}/{coll}_update_errors.txt'
gocfl = f'{coll}/{files}/gocfl'
Path(f'{gocfl}').mkdir(parents=True, exist_ok=True)

# read line from signaturesfile
with open(f_signatures, 'r') as file:
    
    for row in file:
        signature = row.strip()
        print("Signature:",signature)
                
        # create filepaths for metadata, info.json, objects:
        dir_metadata = f'{root}/{metadata}/{signature}/'
        print("           Metadata folder: ",dir_metadata)
        f_infojson = f'{root}/{info}/{signature}.json'        
        print("           Info.json: ",f_infojson)
        dir_sip = f'{root}/{objects}/{signature}/'
        print("           SIP folder: ",dir_sip)
        
        break
        
        # TODO: wander through objects folder to final directory to archive
        # check for empty object path
        if len(os.listdir(dir_sip)) == 0: 
            
            print(f"!!!!!!!!!! Storage root {sip_dir} is empty, check for errors or missing files. ") 
            with open(error_file, 'a') as file:
                file.write(signature)
                print(f"!!!!!!!!!! Signature added to {error_file}")
            continue 
        
        # create storage root for this object
        storage_root = f'{root}/{signature}.zip'
            
        if os.path.exists(storage_root):
            # update procedure necessary, write signature to update file and continue
            print("!!!!!!!!!! Storage root exists already, use gocfl update!")
            with open(f_update, 'a') as file:
                file.write(signature)
                print(f"!!!!!!!!!! Signature added to {update_file}")
            continue    

        else:
            with ZipFile(storage_root, 'w') as zipfile:
                print("           Storage root created: ",storage_root)
            
            # create string
            create_string = f'gocfl create {storage_root} {sip_dir} metadata:{metadata_folder} --config {gocfl_config} -i "{signature}"  --ext-NNNN-metafile-source {info_file}'
            print('\n###############\n',create_string, '\n###############\n')
            create_file = f'{create_file_base}_{signature}.txt'
            with open(create_file, 'w') as file:
                file.write(create_string)
                
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))                

Signature: zhb_10_7891_e-manuscripta-108732
           Metadata folder:  d:/Ingest/zhb_e-manuscripta/metadata/zhb_10_7891_e-manuscripta-108732/
           Info.json:  d:/Ingest/zhb_e-manuscripta/info/zhb_10_7891_e-manuscripta-108732.json
           SIP folder:  d:/Ingest/zhb_e-manuscripta/objects/zhb_10_7891_e-manuscripta-108732/
Finished at  2023-11-10 19:28:02
